In [12]:
# API 
import requests as r
import pandas as pd
from pprint import pprint

# telechargement des mise a jour historique 
from datetime import datetime
from datetime import timedelta

# telechargement des donnes historique
from pathlib import Path

# evite le spam de requete / anti bot
import time
import random

# bar de chargement
from tqdm import tqdm
import threading

meteo en direct

In [27]:
API_KEY = "9c3877cc6c79959b5ebdbbfd6d8c01a4"

url = f"https://api.openweathermap.org/data/2.5/weather?q=Lyon&appid={API_KEY}&units=metric&lang=fr"

data = r.get(url).json()

In [31]:
data

{'coord': {'lon': 4.5833, 'lat': 45.75},
 'weather': [{'id': 804,
   'main': 'Clouds',
   'description': 'couvert',
   'icon': '04d'}],
 'base': 'stations',
 'main': {'temp': 18.03,
  'feels_like': 17.14,
  'temp_min': 15.94,
  'temp_max': 18.05,
  'pressure': 1019,
  'humidity': 48,
  'sea_level': 1019,
  'grnd_level': 960},
 'visibility': 10000,
 'wind': {'speed': 0.45, 'deg': 28, 'gust': 4.92},
 'clouds': {'all': 90},
 'dt': 1781100270,
 'sys': {'type': 2,
  'id': 2007821,
  'country': 'FR',
  'sunrise': 1781063523,
  'sunset': 1781119819},
 'timezone': 7200,
 'id': 2996943,
 'name': 'Lyon',
 'cod': 200}

pollution en direct

In [ ]:
url = f"http://api.openweathermap.org/data/2.5/air_pollution?lat={data['coord']['lat']}&lon={data['coord']['lon']}&appid={API_KEY}"

data_pol = r.get(url).json()

In [30]:
data_pol

{'coord': {'lon': 4.5833, 'lat': 45.75},
 'list': [{'main': {'aqi': 2},
   'components': {'co': 86.34,
    'no': 0.05,
    'no2': 0.23,
    'o3': 72.2,
    'so2': 0.03,
    'pm2_5': 0.5,
    'pm10': 0.5,
    'nh3': 0.78},
   'dt': 1781100461}]}

prediction de la meteo des 5 prochain jour

In [33]:
url = f"https://api.openweathermap.org/data/2.5/forecast?lat={data['coord']['lat']}&lon={data['coord']['lon']}&appid={API_KEY}"

data_pred = r.get(url).json()

In [45]:
pprint(data_pred)

{'city': {'coord': {'lat': 45.75, 'lon': 4.5833},
          'country': 'FR',
          'id': 2996943,
          'name': 'Arrondissement de Lyon',
          'population': 1489152,
          'sunrise': 1781063523,
          'sunset': 1781119819,
          'timezone': 7200},
 'cnt': 40,
 'cod': '200',
 'list': [{'clouds': {'all': 70},
           'dt': 1781103600,
           'dt_txt': '2026-06-10 15:00:00',
           'main': {'feels_like': 289.72,
                    'grnd_level': 960,
                    'humidity': 49,
                    'pressure': 1020,
                    'sea_level': 1020,
                    'temp': 290.64,
                    'temp_kf': -3,
                    'temp_max': 293.64,
                    'temp_min': 290.64},
           'pop': 0,
           'sys': {'pod': 'd'},
           'visibility': 10000,
           'weather': [{'description': 'broken clouds',
                        'icon': '04d',
                        'id': 803,
                        'main': 

Collecte et Creation de notre base de donnees

In [ ]:
url_historique = f"https://archive-api.open-meteo.com/v1/archive?latitude=45.7491&longitude=4.8479&start_date=1960-06-11&end_date=2026-06-10&daily=weather_code,temperature_2m_mean,temperature_2m_min,temperature_2m_max,wind_speed_10m_max,wind_gusts_10m_max,rain_sum,snowfall_sum,precipitation_hours,apparent_temperature_mean,apparent_temperature_max,apparent_temperature_min,shortwave_radiation_sum,et0_fao_evapotranspiration,cloud_cover_mean,relative_humidity_2m_mean,snowfall_water_equivalent_sum,pressure_msl_mean,surface_pressure_mean,soil_temperature_0_to_100cm_mean"

In [3]:
data_hist = r.get(url_historique).json()

In [16]:
pprint(data_hist.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])


In [13]:
pprint(data_hist['daily'].keys())

dict_keys(['time', 'weather_code', 'temperature_2m_mean', 'temperature_2m_min', 'temperature_2m_max', 'wind_speed_10m_max', 'wind_gusts_10m_max', 'rain_sum', 'snowfall_sum', 'precipitation_hours', 'apparent_temperature_mean', 'apparent_temperature_max', 'apparent_temperature_min', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration', 'cloud_cover_mean', 'relative_humidity_2m_mean', 'snowfall_water_equivalent_sum', 'pressure_msl_mean', 'surface_pressure_mean', 'soil_temperature_0_to_100cm_mean'])


In [ ]:
df = pd.DataFrame.from_dict(data_hist['daily'])
df

,time,weather_code,temperature_2m_mean,temperature_2m_min,temperature_2m_max,wind_speed_10m_max,wind_gusts_10m_max,rain_sum,snowfall_sum,precipitation_hours,...,apparent_temperature_max,apparent_temperature_min,shortwave_radiation_sum,et0_fao_evapotranspiration,cloud_cover_mean,relative_humidity_2m_mean,snowfall_water_equivalent_sum,pressure_msl_mean,surface_pressure_mean,soil_temperature_0_to_100cm_mean
0,1960-06-11,51,17.6,12.9,22.2,14.7,31.3,0.5,0.0,3.0,...,19.9,12.8,21.37,4.09,47,68,0.0,1021.4,1000.6,17.4
1,1960-06-12,2,19.8,12.4,25.6,11.9,28.8,0.0,0.0,0.0,...,27.4,11.7,28.13,5.17,5,65,0.0,1017.9,997.2,17.5
2,1960-06-13,55,20.0,16.1,26.8,14.1,35.3,6.0,0.0,11.0,...,25.5,15.0,20.06,4.29,46,68,0.0,1012.4,991.9,17.8
3,1960-06-14,3,16.9,13.4,21.1,14.8,33.8,0.0,0.0,0.0,...,18.9,12.0,24.56,4.52,53,61,0.0,1018.2,997.3,17.5
4,1960-06-15,3,16.9,10.5,21.9,18.4,41.8,0.0,0.0,0.0,...,19.3,8.8,25.00,4.91,55,59,0.0,1023.6,1002.7,17.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24101,2026-06-06,51,18.3,11.1,24.0,9.8,27.0,0.8,0.0,5.0,...,24.8,10.8,27.70,4.94,36,65,0.0,1017.9,997.2,19.0
24102,2026-06-07,3,19.3,13.9,24.0,9.7,25.2,0.0,0.0,0.0,...,24.9,13.1,29.59,5.21,35,57,0.0,1023.4,1002.6,19.5
24103,2026-06-08,53,20.4,14.1,27.0,11.3,33.8,4.5,0.0,10.0,...,28.8,14.4,22.06,4.25,52,73,0.0,1018.4,997.7,19.5
24104,2026-06-09,53,17.5,14.9,19.8,13.1,34.2,1.1,0.0,3.0,...,19.9,13.4,17.51,3.55,92,66,0.0,1019.6,998.8,19.4


poids de la requete en nombre de call 3 440 calls

In [29]:
df.to_csv("historique.csv", index=False)

Automatisationde notre collecte de donnes

In [8]:
end_date = (datetime.today()- timedelta(days=1)).strftime("%Y-%m-%d")
start_date =  (datetime.today() - timedelta(days=8)).strftime("%Y-%m-%d")
print(end_date, start_date)

2026-06-12 2026-06-05


poids de la requete en nombre de call 2 calls

In [9]:
end_date = (datetime.today()- timedelta(days=1)).strftime("%Y-%m-%d")
start_date =  (datetime.today() - timedelta(days=8)).strftime("%Y-%m-%d")

url_maj = f"https://archive-api.open-meteo.com/v1/archive?latitude=45.7491&longitude=4.8479&start_date={start_date}&end_date={end_date}&daily=weather_code,temperature_2m_mean,temperature_2m_min,temperature_2m_max,wind_speed_10m_max,wind_gusts_10m_max,rain_sum,snowfall_sum,precipitation_hours,apparent_temperature_mean,apparent_temperature_max,apparent_temperature_min,shortwave_radiation_sum,et0_fao_evapotranspiration,cloud_cover_mean,relative_humidity_2m_mean,snowfall_water_equivalent_sum,pressure_msl_mean,surface_pressure_mean,soil_temperature_0_to_100cm_mean"

data_maj = r.get(url_maj).json()

df_maj = pd.DataFrame(data_maj['daily'])

df_maj

,time,weather_code,temperature_2m_mean,temperature_2m_min,temperature_2m_max,wind_speed_10m_max,wind_gusts_10m_max,rain_sum,snowfall_sum,precipitation_hours,...,apparent_temperature_max,apparent_temperature_min,shortwave_radiation_sum,et0_fao_evapotranspiration,cloud_cover_mean,relative_humidity_2m_mean,snowfall_water_equivalent_sum,pressure_msl_mean,surface_pressure_mean,soil_temperature_0_to_100cm_mean
0,2026-06-05,51,15.3,11.4,19.5,10.6,28.1,0.8,0.0,5.0,...,18.5,11.1,23.12,3.90,62,68,0.0,1016.5,995.5,19.0
1,2026-06-06,51,18.3,11.1,24.0,9.8,27.0,0.8,0.0,5.0,...,24.8,10.8,27.70,4.94,36,65,0.0,1017.9,997.2,19.0
2,2026-06-07,3,19.3,13.9,24.0,9.7,25.2,0.0,0.0,0.0,...,24.9,13.1,29.59,5.21,35,57,0.0,1023.4,1002.6,19.5
3,2026-06-08,53,20.4,14.1,27.0,11.3,33.8,4.5,0.0,10.0,...,28.8,14.4,22.06,4.25,52,73,0.0,1018.4,997.7,19.5
4,2026-06-09,53,17.5,14.9,19.8,13.1,34.2,1.1,0.0,3.0,...,19.9,13.4,17.51,3.55,92,66,0.0,1019.6,998.8,19.4
5,2026-06-10,51,16.2,11.6,19.6,15.4,39.2,0.3,0.0,2.0,...,17.6,10.2,24.69,4.39,45,57,0.0,1020.2,999.3,18.9
6,2026-06-11,3,17.5,11.9,22.6,13.5,34.2,0.0,0.0,0.0,...,19.3,10.7,23.57,4.58,49,52,0.0,1024.3,1003.4,19.1
7,2026-06-12,3,20.7,12.1,27.0,15.9,40.3,0.0,0.0,0.0,...,24.8,11.5,25.92,5.45,27,53,0.0,1026.4,1005.7,19.7


In [11]:
df_maj.to_csv("test.csv", index=False)

In [10]:
df_maj = pd.DataFrame(data_maj['daily'])

df_maj

,time,weather_code,temperature_2m_mean,temperature_2m_min,temperature_2m_max,wind_speed_10m_max,wind_gusts_10m_max,rain_sum,snowfall_sum,precipitation_hours,...,apparent_temperature_max,apparent_temperature_min,shortwave_radiation_sum,et0_fao_evapotranspiration,cloud_cover_mean,relative_humidity_2m_mean,snowfall_water_equivalent_sum,pressure_msl_mean,surface_pressure_mean,soil_temperature_0_to_100cm_mean
0,2026-06-05,51,15.3,11.4,19.5,10.6,28.1,0.8,0.0,5.0,...,18.5,11.1,23.12,3.90,62,68,0.0,1016.5,995.5,19.0
1,2026-06-06,51,18.3,11.1,24.0,9.8,27.0,0.8,0.0,5.0,...,24.8,10.8,27.70,4.94,36,65,0.0,1017.9,997.2,19.0
2,2026-06-07,3,19.3,13.9,24.0,9.7,25.2,0.0,0.0,0.0,...,24.9,13.1,29.59,5.21,35,57,0.0,1023.4,1002.6,19.5
3,2026-06-08,53,20.4,14.1,27.0,11.3,33.8,4.5,0.0,10.0,...,28.8,14.4,22.06,4.25,52,73,0.0,1018.4,997.7,19.5
4,2026-06-09,53,17.5,14.9,19.8,13.1,34.2,1.1,0.0,3.0,...,19.9,13.4,17.51,3.55,92,66,0.0,1019.6,998.8,19.4
5,2026-06-10,51,16.2,11.6,19.6,15.4,39.2,0.3,0.0,2.0,...,17.6,10.2,24.69,4.39,45,57,0.0,1020.2,999.3,18.9
6,2026-06-11,3,17.5,11.9,22.6,13.5,34.2,0.0,0.0,0.0,...,19.3,10.7,23.57,4.58,49,52,0.0,1024.3,1003.4,19.1
7,2026-06-12,3,20.7,12.1,27.0,15.9,40.3,0.0,0.0,0.0,...,24.8,11.5,25.92,5.45,27,53,0.0,1026.4,1005.7,19.7


concat les 2 data frame

In [6]:
df_historique = pd.read_csv("historique.csv")

In [12]:
df_maj = pd.read_csv("test.csv")

In [13]:
df_total = pd.concat([df_historique, df_maj])
df_total

,time,weather_code,temperature_2m_mean,temperature_2m_min,temperature_2m_max,wind_speed_10m_max,wind_gusts_10m_max,rain_sum,snowfall_sum,precipitation_hours,...,apparent_temperature_max,apparent_temperature_min,shortwave_radiation_sum,et0_fao_evapotranspiration,cloud_cover_mean,relative_humidity_2m_mean,snowfall_water_equivalent_sum,pressure_msl_mean,surface_pressure_mean,soil_temperature_0_to_100cm_mean
0,1960-06-11,51,17.6,12.9,22.2,14.7,31.3,0.5,0.0,3.0,...,19.9,12.8,21.37,4.09,47,68,0.0,1021.4,1000.6,17.4
1,1960-06-12,2,19.8,12.4,25.6,11.9,28.8,0.0,0.0,0.0,...,27.4,11.7,28.13,5.17,5,65,0.0,1017.9,997.2,17.5
2,1960-06-13,55,20.0,16.1,26.8,14.1,35.3,6.0,0.0,11.0,...,25.5,15.0,20.06,4.29,46,68,0.0,1012.4,991.9,17.8
3,1960-06-14,3,16.9,13.4,21.1,14.8,33.8,0.0,0.0,0.0,...,18.9,12.0,24.56,4.52,53,61,0.0,1018.2,997.3,17.5
4,1960-06-15,3,16.9,10.5,21.9,18.4,41.8,0.0,0.0,0.0,...,19.3,8.8,25.00,4.91,55,59,0.0,1023.6,1002.7,17.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3,2026-06-08,53,20.4,14.1,27.0,11.3,33.8,4.5,0.0,10.0,...,28.8,14.4,22.06,4.25,52,73,0.0,1018.4,997.7,19.5
4,2026-06-09,53,17.5,14.9,19.8,13.1,34.2,1.1,0.0,3.0,...,19.9,13.4,17.51,3.55,92,66,0.0,1019.6,998.8,19.4
5,2026-06-10,51,16.2,11.6,19.6,15.4,39.2,0.3,0.0,2.0,...,17.6,10.2,24.69,4.39,45,57,0.0,1020.2,999.3,18.9
6,2026-06-11,3,17.5,11.9,22.6,13.5,34.2,0.0,0.0,0.0,...,19.3,10.7,23.57,4.58,49,52,0.0,1024.3,1003.4,19.1


Determiner les villes a prendre

liste de 352 villes 

In [14]:
villes_mondiales = [
    # Europe
    "Paris", "Londres", "Berlin", "Madrid", "Rome", "Bruxelles", "Amsterdam",
    "Vienne", "Prague", "Varsovie", "Athènes", "Lisbonne", "Dublin",
    "Copenhague", "Stockholm", "Oslo", "Helsinki", "Reykjavik",
    "Budapest", "Bucarest", "Sofia", "Belgrade", "Zagreb", "Ljubljana",
    "Sarajevo", "Skopje", "Tirana", "Podgorica", "Chisinau",
    "Kyiv", "Moscou", "Saint-Pétersbourg", "Minsk", "Vilnius",
    "Riga", "Tallinn", "Luxembourg", "Monaco", "Andorre-la-Vieille",
    "Saint-Marin", "La Valette", "Genève", "Zurich", "Bâle",
    "Munich", "Hambourg", "Francfort", "Cologne", "Barcelone",
    "Valence", "Séville", "Milan", "Naples", "Turin",
    "Manchester", "Birmingham", "Glasgow", "Édimbourg",

    # Amérique du Nord
    "New York", "Washington", "Los Angeles", "Chicago", "Houston",
    "Phoenix", "Philadelphie", "San Antonio", "San Diego", "Dallas",
    "San Jose", "Austin", "Jacksonville", "San Francisco", "Seattle",
    "Boston", "Denver", "Miami", "Atlanta", "Detroit",
    "Toronto", "Montréal", "Vancouver", "Calgary", "Ottawa",
    "Edmonton", "Winnipeg", "Québec", "Mexico", "Guadalajara",
    "Monterrey", "Puebla", "Tijuana", "Cancún", "Mérida",

    # Amérique centrale et Caraïbes
    "La Havane", "Santo Domingo", "Port-au-Prince", "San Juan",
    "Kingston", "Nassau", "Bridgetown", "Port of Spain",
    "Belmopan", "Belize City", "Guatemala", "San Salvador",
    "Tegucigalpa", "Managua", "San José", "Panama",

    # Amérique du Sud
    "Bogotá", "Medellín", "Cali", "Cartagena",
    "Caracas", "Maracaibo",
    "Quito", "Guayaquil",
    "Lima", "Arequipa", "Cusco",
    "La Paz", "Santa Cruz de la Sierra",
    "Asunción",
    "Santiago", "Valparaíso",
    "Buenos Aires", "Córdoba", "Rosario", "Mendoza",
    "Montevideo",
    "São Paulo", "Rio de Janeiro", "Brasília", "Salvador",
    "Fortaleza", "Recife", "Belo Horizonte", "Curitiba",
    "Porto Alegre", "Manaus", "Belém", "Goiânia",

    # Afrique du Nord
    "Le Caire", "Alexandrie",
    "Casablanca", "Rabat", "Marrakech", "Fès", "Tanger",
    "Alger", "Oran", "Constantine",
    "Tunis", "Sfax",
    "Tripoli", "Benghazi",

    # Afrique de l'Ouest
    "Dakar", "Abidjan", "Yamoussoukro", "Accra", "Kumasi",
    "Lagos", "Abuja", "Kano", "Port Harcourt",
    "Cotonou", "Porto-Novo", "Lomé",
    "Conakry", "Freetown", "Monrovia",
    "Bamako", "Ouagadougou", "Niamey",
    "Bissau", "Praia", "Nouakchott",

    # Afrique centrale
    "Kinshasa", "Lubumbashi",
    "Brazzaville",
    "Libreville",
    "Yaoundé", "Douala",
    "Bangui",
    "N'Djamena",
    "Malabo",

    # Afrique de l'Est
    "Nairobi", "Mombasa",
    "Kampala",
    "Kigali",
    "Bujumbura",
    "Addis-Abeba",
    "Djibouti",
    "Mogadiscio",
    "Khartoum",
    "Juba",
    "Dar es Salaam", "Dodoma",
    "Zanzibar",

    # Afrique australe
    "Johannesburg", "Le Cap", "Durban", "Pretoria",
    "Bloemfontein",
    "Gaborone",
    "Windhoek",
    "Lusaka",
    "Harare",
    "Maputo",
    "Lilongwe",
    "Antananarivo",
    "Port-Louis",

    # Moyen-Orient
    "Istanbul", "Ankara",
    "Jérusalem", "Tel-Aviv",
    "Amman",
    "Beyrouth",
    "Damas",
    "Bagdad", "Bassorah",
    "Koweït",
    "Riyad", "Djeddah", "La Mecque", "Médine",
    "Manama",
    "Doha",
    "Abou Dabi", "Dubaï", "Sharjah",
    "Mascate",
    "Sanaa", "Aden",
    "Téhéran", "Ispahan", "Machhad", "Chiraz",

    # Asie du Sud
    "Delhi", "Mumbai", "Bangalore", "Hyderabad",
    "Chennai", "Kolkata", "Pune", "Ahmedabad",
    "Jaipur", "Lucknow", "Surat",
    "Karachi", "Lahore", "Islamabad", "Rawalpindi",
    "Dhaka", "Chittagong",
    "Katmandou",
    "Thimphou",
    "Colombo",
    "Malé",

    # Asie du Sud-Est
    "Bangkok", "Chiang Mai",
    "Hanoï", "Hô Chi Minh-Ville", "Da Nang",
    "Phnom Penh",
    "Vientiane",
    "Yangon", "Naypyidaw",
    "Kuala Lumpur", "George Town", "Johor Bahru",
    "Singapour",
    "Jakarta", "Surabaya", "Bandung", "Medan",
    "Denpasar",
    "Manille", "Cebu", "Davao",
    "Bandar Seri Begawan",
    "Dili",

    # Chine
    "Pékin", "Shanghai", "Shenzhen", "Guangzhou",
    "Hong Kong", "Macao", "Tianjin",
    "Chongqing", "Chengdu", "Wuhan",
    "Nankin", "Hangzhou", "Suzhou",
    "Xi'an", "Qingdao", "Harbin",
    "Shenyang", "Kunming", "Xiamen",
    "Ningbo", "Fuzhou", "Changsha",
    "Zhengzhou", "Jinan", "Urumqi",

    # Japon
    "Tokyo", "Osaka", "Kyoto", "Nagoya",
    "Yokohama", "Kobe", "Sapporo",
    "Fukuoka", "Hiroshima", "Sendai",

    # Corée
    "Séoul", "Busan", "Incheon",
    "Daegu", "Daejeon", "Gwangju",

    # Taïwan
    "Taipei", "Kaohsiung", "Taichung",

    # Mongolie
    "Oulan-Bator",

    # Asie centrale
    "Astana", "Almaty",
    "Tachkent", "Samarcande",
    "Bichkek",
    "Douchanbé",
    "Achgabat",

    # Océanie
    "Sydney", "Melbourne", "Brisbane",
    "Perth", "Adélaïde", "Canberra",
    "Gold Coast", "Hobart", "Darwin",
    "Auckland", "Wellington", "Christchurch",
    "Suva", "Port Moresby",
    "Apia", "Nuku'alofa",
    "Port-Vila", "Honiara"
]

In [16]:
len(villes_mondiales)

352

liste de 15 villes

In [5]:
villes_monde = [
    "New York",
    "São Paulo",
    "Londres",
    "Paris",
    "Moscou",
    "Le Caire",
    "Lagos",
    "Dubaï",
    "Mumbai",
    "Pékin",
    "Shanghai",
    "Tokyo",
    "Séoul",
    "Singapour",
    "Sydney"
]

In [6]:
len(villes_monde)

15

liste de 50 villes 

In [11]:
cities = [
    # Europe
    ("London", 51.5074, -0.1278),
    ("Paris", 48.8566, 2.3522),
    ("Madrid", 40.4168, -3.7038),
    ("Rome", 41.9028, 12.4964),
    ("Berlin", 52.5200, 13.4050),
    ("Moscow", 55.7558, 37.6176),
    ("Stockholm", 59.3293, 18.0686),

    # North America
    ("New York", 40.7128, -74.0060),
    ("Chicago", 41.8781, -87.6298),
    ("Los Angeles", 34.0522, -118.2437),
    ("Mexico City", 19.4326, -99.1332),
    ("Toronto", 43.6532, -79.3832),
    ("Anchorage", 61.2181, -149.9003),

    # South America
    ("Sao Paulo", -23.5505, -46.6333),
    ("Rio de Janeiro", -22.9068, -43.1729),
    ("Buenos Aires", -34.6037, -58.3816),
    ("Santiago", -33.4489, -70.6693),
    ("Lima", -12.0464, -77.0428),
    ("Bogota", 4.7110, -74.0721),

    # Africa
    ("Cairo", 30.0444, 31.2357),
    ("Casablanca", 33.5731, -7.5898),
    ("Lagos", 6.5244, 3.3792),
    ("Nairobi", -1.2921, 36.8219),
    ("Johannesburg", -26.2041, 28.0473),
    ("Cape Town", -33.9249, 18.4241),

    # Middle East
    ("Dubai", 25.2048, 55.2708),
    ("Riyadh", 24.7136, 46.6753),
    ("Tehran", 35.6892, 51.3890),
    ("Jerusalem", 31.7683, 35.2137),

    # South Asia
    ("Mumbai", 19.0760, 72.8777),
    ("Delhi", 28.6139, 77.2090),
    ("Karachi", 24.8607, 67.0011),
    ("Dhaka", 23.8103, 90.4125),
    ("Kathmandu", 27.7172, 85.3240),

    # East Asia
    ("Beijing", 39.9042, 116.4074),
    ("Shanghai", 31.2304, 121.4737),
    ("Tokyo", 35.6762, 139.6503),
    ("Seoul", 37.5665, 126.9780),
    ("Hong Kong", 22.3193, 114.1694),

    # Southeast Asia
    ("Bangkok", 13.7563, 100.5018),
    ("Singapore", 1.3521, 103.8198),
    ("Jakarta", -6.2088, 106.8456),
    ("Manila", 14.5995, 120.9842),

    # Oceania
    ("Sydney", -33.8688, 151.2093),
    ("Melbourne", -37.8136, 144.9631),
    ("Perth", -31.9505, 115.8605),
    ("Auckland", -36.8509, 174.7645),

    # Extreme climates
    ("Reykjavik", 64.1466, -21.9426),
    ("Ushuaia", -54.8019, -68.3030),
    ("Fairbanks", 64.8378, -147.7164)
]

In [4]:
len(cities)

50

programme pour automatiser la collecte des donnees 

In [13]:
cities = [
    # Europe   lat      lon
    ("London", 51.5074, -0.1278),
    ("Paris", 48.8566, 2.3522),
    ("Madrid", 40.4168, -3.7038),
    ("Rome", 41.9028, 12.4964),
    ("Berlin", 52.5200, 13.4050),
    ("Moscow", 55.7558, 37.6176),
    ("Stockholm", 59.3293, 18.0686),

    # North America
    ("New York", 40.7128, -74.0060),
    ("Chicago", 41.8781, -87.6298),
    ("Los Angeles", 34.0522, -118.2437),
    ("Mexico City", 19.4326, -99.1332),
    ("Toronto", 43.6532, -79.3832),
    ("Anchorage", 61.2181, -149.9003),

    # South America
    ("Sao Paulo", -23.5505, -46.6333),
    ("Rio de Janeiro", -22.9068, -43.1729),
    ("Buenos Aires", -34.6037, -58.3816),
    ("Santiago", -33.4489, -70.6693),
    ("Lima", -12.0464, -77.0428),
    ("Bogota", 4.7110, -74.0721),

    # Africa
    ("Cairo", 30.0444, 31.2357),
    ("Casablanca", 33.5731, -7.5898),
    ("Lagos", 6.5244, 3.3792),
    ("Nairobi", -1.2921, 36.8219),
    ("Johannesburg", -26.2041, 28.0473),
    ("Cape Town", -33.9249, 18.4241),

    # Middle East
    ("Dubai", 25.2048, 55.2708),
    ("Riyadh", 24.7136, 46.6753),
    ("Tehran", 35.6892, 51.3890),
    ("Jerusalem", 31.7683, 35.2137),

    # South Asia
    ("Mumbai", 19.0760, 72.8777),
    ("Delhi", 28.6139, 77.2090),
    ("Karachi", 24.8607, 67.0011),
    ("Dhaka", 23.8103, 90.4125),
    ("Kathmandu", 27.7172, 85.3240),

    # East Asia
    ("Beijing", 39.9042, 116.4074),
    ("Shanghai", 31.2304, 121.4737),
    ("Tokyo", 35.6762, 139.6503),
    ("Seoul", 37.5665, 126.9780),
    ("Hong Kong", 22.3193, 114.1694),

    # Southeast Asia
    ("Bangkok", 13.7563, 100.5018),
    ("Singapore", 1.3521, 103.8198),
    ("Jakarta", -6.2088, 106.8456),
    ("Manila", 14.5995, 120.9842),

    # Oceania
    ("Sydney", -33.8688, 151.2093),
    ("Melbourne", -37.8136, 144.9631),
    ("Perth", -31.9505, 115.8605),
    ("Auckland", -36.8509, 174.7645),

    # Extreme climates
    ("Reykjavik", 64.1466, -21.9426),
    ("Ushuaia", -54.8019, -68.3030),
    ("Fairbanks", 64.8378, -147.7164)
]

for city, lat, lon in cities:

    path = Path(f"hist_villes/{city}_hist_1960_2026.parquet")

    if path.exists():
        pprint(f"  ✓   file {city} already exist")
        continue

    url_historique = (f"https://archive-api.open-meteo.com/v1/archive?"
                    f"latitude={lat}"
                    f"&longitude={lon}"
                    "&start_date=1960-06-11"
                    "&end_date=2026-06-10"
                    "&daily=weather_code,"
                    "temperature_2m_mean,"
                    "temperature_2m_min,"
                    "temperature_2m_max,"
                    "wind_speed_10m_max,"
                    "wind_gusts_10m_max,"
                    "rain_sum,snowfall_sum,"
                    "precipitation_hours,"
                    "apparent_temperature_mean,"
                    "apparent_temperature_max,"
                    "apparent_temperature_min,"
                    "shortwave_radiation_sum,"
                    "et0_fao_evapotranspiration,"
                    "cloud_cover_mean,"
                    "relative_humidity_2m_mean,"
                    "snowfall_water_equivalent_sum,"
                    "pressure_msl_mean,"
                    "surface_pressure_mean,"
                    "soil_temperature_0_to_100cm_mean")
    
    try:
        done = False

        def progress():
            with tqdm(total=150, desc=city, leave=False) as bar:
                while not done and bar.n < 160:
                    time.sleep(1)
                    bar.update(1)
                
                if done:
                    bar.n = bar.total
                    bar.refresh()
                    pprint("")

        
        threading.Thread(target=progress, daemon=True).start()
        
        response = r.get(url_historique, timeout=240)
        response.raise_for_status()

        hist_ville = response.json()

        pd.DataFrame(hist_ville["daily"]).to_parquet(path, index=False)

        pprint(f"✓ {city}")

    except Exception as erreur:
        pprint(f"✗ {city} : {erreur}/n")
    
    finally:
        done = True

    time.sleep(3 + random.random())




'  ✓   file London already exist'
'  ✓   file Paris already exist'
'  ✓   file Madrid already exist'
'  ✓   file Rome already exist'
'  ✓   file Berlin already exist'
'  ✓   file Moscow already exist'
'  ✓   file Stockholm already exist'
'  ✓   file New York already exist'
'  ✓   file Chicago already exist'
'  ✓   file Los Angeles already exist'
'  ✓   file Mexico City already exist'
'  ✓   file Toronto already exist'
'  ✓   file Anchorage already exist'
'  ✓   file Sao Paulo already exist'
'  ✓   file Rio de Janeiro already exist'
'  ✓   file Buenos Aires already exist'
'  ✓   file Santiago already exist'
'  ✓   file Lima already exist'
'  ✓   file Bogota already exist'
'  ✓   file Cairo already exist'
'  ✓   file Casablanca already exist'
'  ✓   file Lagos already exist'
'  ✓   file Nairobi already exist'
'  ✓   file Johannesburg already exist'
'  ✓   file Cape Town already exist'
'  ✓   file Dubai already exist'
'  ✓   file Riyadh already exist'
'  ✓   file Tehran already exist'
'  

Ushuaia:  22%|██▏       | 33/150 [00:33<01:58,  1.01s/it]

'✓ Ushuaia'


''


'✓ Fairbanks'


Code pour mettre les donnes de toute les villes a jour

In [ ]:
cities = [
    # Europe   lat      lon
    ("London", 51.5074, -0.1278),
    ("Paris", 48.8566, 2.3522),
    ("Madrid", 40.4168, -3.7038),
    ("Rome", 41.9028, 12.4964),
    ("Berlin", 52.5200, 13.4050),
    ("Moscow", 55.7558, 37.6176),
    ("Stockholm", 59.3293, 18.0686),

    # North America
    ("New York", 40.7128, -74.0060),
    ("Chicago", 41.8781, -87.6298),
    ("Los Angeles", 34.0522, -118.2437),
    ("Mexico City", 19.4326, -99.1332),
    ("Toronto", 43.6532, -79.3832),
    ("Anchorage", 61.2181, -149.9003),

    # South America
    ("Sao Paulo", -23.5505, -46.6333),
    ("Rio de Janeiro", -22.9068, -43.1729),
    ("Buenos Aires", -34.6037, -58.3816),
    ("Santiago", -33.4489, -70.6693),
    ("Lima", -12.0464, -77.0428),
    ("Bogota", 4.7110, -74.0721),

    # Africa
    ("Cairo", 30.0444, 31.2357),
    ("Casablanca", 33.5731, -7.5898),
    ("Lagos", 6.5244, 3.3792),
    ("Nairobi", -1.2921, 36.8219),
    ("Johannesburg", -26.2041, 28.0473),
    ("Cape Town", -33.9249, 18.4241),

    # Middle East
    ("Dubai", 25.2048, 55.2708),
    ("Riyadh", 24.7136, 46.6753),
    ("Tehran", 35.6892, 51.3890),
    ("Jerusalem", 31.7683, 35.2137),

    # South Asia
    ("Mumbai", 19.0760, 72.8777),
    ("Delhi", 28.6139, 77.2090),
    ("Karachi", 24.8607, 67.0011),
    ("Dhaka", 23.8103, 90.4125),
    ("Kathmandu", 27.7172, 85.3240),

    # East Asia
    ("Beijing", 39.9042, 116.4074),
    ("Shanghai", 31.2304, 121.4737),
    ("Tokyo", 35.6762, 139.6503),
    ("Seoul", 37.5665, 126.9780),
    ("Hong Kong", 22.3193, 114.1694),

    # Southeast Asia
    ("Bangkok", 13.7563, 100.5018),
    ("Singapore", 1.3521, 103.8198),
    ("Jakarta", -6.2088, 106.8456),
    ("Manila", 14.5995, 120.9842),

    # Oceania
    ("Sydney", -33.8688, 151.2093),
    ("Melbourne", -37.8136, 144.9631),
    ("Perth", -31.9505, 115.8605),
    ("Auckland", -36.8509, 174.7645),

    # Extreme climates
    ("Reykjavik", 64.1466, -21.9426),
    ("Ushuaia", -54.8019, -68.3030),
    ("Fairbanks", 64.8378, -147.7164)
]

end_date = (datetime.today()- timedelta(days=1)).strftime("%Y-%m-%d")
start_date =  (datetime.today() - timedelta(days=8)).strftime("%Y-%m-%d")


for city, lat, lon in cities:

    path = Path(f"hist_villes/{city}_hist_1960_{end_date}.parquet")

    if path.exists():
        pprint(f"  ✓   file {city} already up to date")
        continue

# modifier la start date apres la premier execution 

    url_historique = (f"https://archive-api.open-meteo.com/v1/archive?"
                    f"latitude={lat}"
                    f"&longitude={lon}"
                    "&start_date=2026-06-10"
                    f"&end_date={end_date}"
                    "&daily=weather_code,"
                    "temperature_2m_mean,"
                    "temperature_2m_min,"
                    "temperature_2m_max,"
                    "wind_speed_10m_max,"
                    "wind_gusts_10m_max,"
                    "rain_sum,snowfall_sum,"
                    "precipitation_hours,"
                    "apparent_temperature_mean,"
                    "apparent_temperature_max,"
                    "apparent_temperature_min,"
                    "shortwave_radiation_sum,"
                    "et0_fao_evapotranspiration,"
                    "cloud_cover_mean,"
                    "relative_humidity_2m_mean,"
                    "snowfall_water_equivalent_sum,"
                    "pressure_msl_mean,"
                    "surface_pressure_mean,"
                    "soil_temperature_0_to_100cm_mean")
    
    try:
        done = False

        def progress():
            with tqdm(total=5, desc=city, leave=False) as bar:
                while not done and bar.n < 160:
                    time.sleep(1)
                    bar.update(1)
                
                if done:
                    bar.n = bar.total
                    bar.refresh()
                    pprint("")

        
        threading.Thread(target=progress, daemon=True).start()
        
        response = r.get(url_historique, timeout=240)
        response.raise_for_status()

        maj = response.json()

        maj = pd.DataFrame(maj["daily"])
        # modifier nom du fichier apres premier execution 
        hist = pd.read_parquet(f"hist_villes/{city}_hist_1960_2026.parquet")
        
        final = pd.concat([maj, hist], ignore_index=True)
        final.to_parquet(path, index=False)

        pprint(f"✓ {city} up to date")

    except Exception as erreur:
        pprint(f"✗ {city} : {erreur}/n")
    
    finally:
        done = True

    time.sleep(3 + random.random())

London:   4%|▍         | 4/100 [00:04<01:42,  1.06s/it]

'✓ London up to date'


''


Paris:   0%|          | 0/100 [00:00<?, ?it/s]

'✓ Paris up to date'


...

'✓ Stockholm up to date'


''


New York:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

''
